# VidTranscribe.ai - Colab Evaluation Pipeline

Notebook này chạy bộ benchmark mới nhất trên Google Colab:

1. Mount Google Drive
2. Clone/Pull repo
3. Cài FFmpeg, Ollama, Python dependencies
4. Kiểm tra dataset 100 câu: 70 PhoST + 30 ViMedCSS
5. Chạy Benchmark 3: Gemini 2.5 LLM Judge, 20 câu/batch, chạy song song theo batch
6. Chạy Performance benchmark
7. Chạy Group-level Sync benchmark
8. Copy kết quả về Google Drive


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone/Pull Repository

In [ ]:
import os
REPO_URL = 'https://github.com/HoangKhang226/VidTranscribe.ai.git'
REPO_DIR = '/content/VidTranscribe.ai'

if os.path.exists(REPO_DIR):
    %cd /content/VidTranscribe.ai
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd /content/VidTranscribe.ai

## 3. Cài system dependencies: FFmpeg + Ollama

In [ ]:
# FFmpeg cho xử lý audio/video
!apt-get update -y && apt-get install -y ffmpeg

# Ollama cho LLM local của pipeline dịch chính
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &

import time
time.sleep(8)

# Model nhẹ cho Colab. Nếu repo config dùng model khác, có thể pull thêm model đó.
!ollama pull qwen2.5:3b-instruct-q4_K_M

## 4. Cài Python dependencies

In [ ]:
!pip install -r requirements.txt

## 5. Set Gemini API Key

In [ ]:
import os, getpass
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Nhập GOOGLE_API_KEY cho Gemini judge: ')
print('GOOGLE_API_KEY đã được set:', bool(os.environ.get('GOOGLE_API_KEY')))

## 6. Kiểm tra / build dataset 100 câu
Repo đã có sẵn `evaluation/test_cases.json`. Cell này kiểm tra nếu thiếu hoặc không đủ 100 câu thì build lại từ PhoST local + ViMedCSS streaming text-only.

In [ ]:
import json, os
cases_path = 'evaluation/test_cases.json'
need_build = True
if os.path.exists(cases_path):
    with open(cases_path, 'r', encoding='utf-8') as f:
        cases = json.load(f)
    print('Số case hiện có:', len(cases))
    need_build = len(cases) < 100

if need_build:
    !python evaluation/build_hybrid_dataset.py --phost-limit 70 --stress-limit 30 --output evaluation/test_cases.json

with open(cases_path, 'r', encoding='utf-8') as f:
    cases = json.load(f)
print('Dataset final:', len(cases), 'cases')
print('Sample keys:', cases[0].keys())

## 7. Benchmark 3 - Gemini 2.5 LLM Judge

Cấu hình mặc định:
- 100 câu
- 20 câu / batch
- 5 batch song song (`--judge-concurrency 5`)
- structured output bằng LangChain `with_structured_output`

In [ ]:
!python evaluation/benchmark_localization.py \
  --judge \
  --judge-model gemini-2.5-flash-preview-05-20 \
  --judge-batch-size 20 \
  --judge-concurrency 5

## 8. Xem nhanh kết quả Localization Judge

In [ ]:
import pandas as pd, json
df = pd.read_csv('evaluation/localization_results.csv')
display(df[['case_id', 'source', 'domain', 'english_original', 'ai_translation', 'expected_localization', 'ai_score', 'ai_reason']].head(20))
with open('evaluation/localization_results.json', 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

## 9. Benchmark 1 - Performance & Hardware

In [ ]:
# Chạy end-to-end để sinh output video + temp_segments cho sync benchmark
!python evaluation/benchmark_perf.py --mode end_to_end --keep-temp-segments --include-nvidia-smi

## 10. Benchmark 2 - Group-level Sync

In [ ]:
!python evaluation/benchmark_sync_group.py

## 11. Copy kết quả về Google Drive

In [ ]:
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/VidTranscribe_Evaluation'
os.makedirs(DRIVE_DIR, exist_ok=True)

files_to_copy = [
    'evaluation/test_cases.json',
    'evaluation/localization_results.csv',
    'evaluation/localization_results.json',
    'evaluation/perf_results.json',
    'evaluation/sync_group_results.csv',
    'evaluation/sync_group_results.json',
    'evaluation/evaluation_results.md',
]

for file in files_to_copy:
    if os.path.exists(file):
        shutil.copy(file, DRIVE_DIR)
        print('Copied:', file)
    else:
        print('Missing:', file)

print('Done. Results saved to:', DRIVE_DIR)